# 04 — Transfer Learning: Symbolic PPO World 1-1 → 1-2

Fine-tune the 1-1 trained PPO model on World 1-2.

**Requires:** `models/symbolic_ppo/final_model` from notebook 03.

In [1]:
# --- Google Colab Setup ---
import os, sys

if os.getenv('COLAB_RELEASE_TAG'):
    !pip install -q Cython setuptools wheel
    !git clone -b hotfix/version1 https://github.com/lmartim4/CSC-52081-ContinousMountainCar.git /content/repo
    %cd /content/repo
    !pip install -r requirements.txt --no-build-isolation
    sys.path.insert(0, '/content/repo')
    import site; SITE = site.getsitepackages()[0]
    !patch --forward -p0 {SITE}/nes_py/_rom.py                   < patches/nes_py_numpy2.patch || true
    !patch --forward -p0 {SITE}/gym/utils/passive_env_checker.py < patches/gym_bool8_numpy2.patch || true
    !patch --forward -p0 {SITE}/gym_super_mario_bros/smb_env.py  < patches/smb_env_numpy2.patch || true
    !sed -i 's/observation, reward, terminated, truncated, info = self.env.step(action)/_result = self.env.step(action); observation, reward, terminated, info = _result[:4]; truncated = _result[4] if len(_result) > 4 else False/' {SITE}/gym/wrappers/time_limit.py
    !git pull
else:
    if os.path.basename(os.getcwd()) == 'notebooks':
        %cd ..

/home/contente/Documents/ENSTA/autonomous/CSC-52081-ContinousMountainCar


In [2]:
import torch
from stable_baselines3 import PPO

from src.wrappers import make_symbolic_vec_env, make_symbolic_env
from src.utils.callbacks import CheckpointAndLogCallback
from src.config import PPOConfig

print(f'Using CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Using CUDA: True
GPU: NVIDIA GeForce GTX 1650


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


## Transfer Learning: World 1-2

Fine-tune the 1-1 trained model on level 1-2.

In [ ]:
# Create env for world 1-2
env_12 = make_symbolic_vec_env(
    env_id='SuperMarioBros-1-2-v3',
    skip=4,
    n_stack=4,
    flatten=True,
    num_envs=8,
)
print(f'Observation space: {env_12.observation_space.shape}')
print(f'Action space: {env_12.action_space.n}')

In [ ]:
# Load 1-1 model and fine-tune on 1-2
from stable_baselines3 import PPO

model_12 = PPO.load(
    'models/symbolic_ppo/final_model',
    env=env_12,
    device='cpu',
    learning_rate=2.5e-4,  # override saved lr
)

callback_12 = CheckpointAndLogCallback(
    save_path='models/symbolic_ppo_1_2',
    save_freq=50_000,
)

print(f'LR: {model_12.learning_rate}')
print('Training on World 1-2 (transfer from 1-1)...')
model_12.learn(
    total_timesteps=2_000_000,
    callback=callback_12,
    log_interval=1,
    tb_log_name='PPO_1_2',
)
model_12.save('models/symbolic_ppo_1_2/final_model')
print('World 1-2 training complete!')

In [ ]:
# Watch: python watch_agent.py --model models/symbolic_ppo_1_2/final_model --env SuperMarioBros-1-2-v3